# Does CLIP's Image Encoder Geometry Predict Cross-Lingual Cultural Alignment Failures?
### Ashwin Kumar, Harini Raj, Pranav Kushare
**Saarland University | HLCV SS26 Project Demo**

---

## Project Overview
This project evaluates zero-shot vision-language models (**OpenCLIP** and **SigLIP 2**) across **10 object classes** (split into Tier 1: Culturally Universal, and Tier 2: Culturally Embedded) and **6 world regions** using the **GeoDE dataset** (3,000 images).

### Key Methodological Corrections Made:
1. **Language Swap**: Replaced German/Hindi with **Spanish/Arabic** because GeoDE has zero images from Germany/India. Spanish matches 4 GeoDE countries (Argentina, Colombia, Mexico, Spain) and Arabic matches 3 (Egypt, Saudi Arabia, UAE).
2. **West Definition**: Redefined `r_West` as **Europe only**, since GeoDE's "Americas" consists of Latin American countries which are not representative of Western-centric web-scraping biases.

This notebook loads the generated results tables, visualizes the key findings from **Q1 (Visual Geometry)**, **Q2 (Alignment Gap)**, and **Q3 (Prompt Intervention)**, and runs a confound regression analysis.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Set working directory to project root
if os.path.exists('../results'):
    os.chdir('..')

print("Current working directory:", os.getcwd())
print("Results files available:", os.listdir('results/tables'))

## 1. Global Pipeline Log & Statistical Summary
Let's load the main execution log `results/log.csv` which lists all statistical test outputs, including Welch t-tests and the Confound OLS Regressions.

In [ ]:
log_df = pd.read_csv('results/log.csv')
# Show Welch t-tests
display(log_df[log_df['test_type'] == 'Welch_ind'][['timestamp', 'phase', 'model', 'language', 'prompt_template', 'metric_name', 'p_value']])

# Show Confound OLS Regression results
print("\n--- Confound OLS Regression Results (Δ_L ~ Intercept + is_Tier2 + cos_en) ---")
display(log_df[log_df['phase'] == 'Q2_Confound_OLS'][['model', 'language', 'metric_name', 'metric_value', 'p_value', 'notes']])

## 2. Q1: Visual Geometry (Metric 1)
**Question:** Does CLIP's image encoder produce higher inter-region feature divergence for culturally embedded concepts (Tier 2) than universal ones (Tier 1)?
Let's look at the cross-region divergence scores: `1 - mean pairwise cosine similarity` across regions.

In [ ]:
q1_df = pd.read_csv('results/tables/q1_image_geometry.csv')
# Mean overall divergence by tier and model
q1_summary = q1_df.groupby(['model', 'tier'])[['div_overall', 'S_west_gap']].mean()
display(q1_summary)

print("\nVisualizing Q1 Figure (Western Coherence Gap):")
display(Image('results/figures/fig2_west_gap.png', width=700))

## 3. Q2: Cross-Lingual Alignment Gap (Metric 2)
**Question:** Is the cross-lingual alignment gap $\Delta_L = \cos(v, t_{EN}) - \cos(v, t_L)$ larger for Tier 2 than Tier 1?

In [ ]:
q2_df = pd.read_csv('results/tables/q2_alignment_gap.csv')
# Look at P1 (Neutral Baseline) mean alignment gaps
q2_summary = q2_df[q2_df['prompt_id'] == 'P1'].groupby(['model', 'language', 'tier'])['mean_delta_L'].mean().unstack()
display(q2_summary)

print("\nVisualizing Q2 Figure (Alignment Gap Distribution):")
display(Image('results/figures/fig3_delta_L_violin.png', width=700))

## 4. Q3: Prompt Intervention (Metric 3)
**Question:** Can culturally-aware prompt templates ($P3$) reduce the alignment gap relative to the neutral baseline ($P1$)?

In [ ]:
q3_df = pd.read_csv('results/tables/q3_prompt_gain.csv')
# Show average prompt gain G(P3) by model, language, and tier
q3_summary = q3_df.groupby(['model', 'language', 'tier'])['gain_P3'].mean().unstack()
display(q3_summary)

print("\nVisualizing Q3 Prompt Gain Comparison (Spanish):")
display(Image('results/figures/fig5_prompt_gain_es.png', width=700))

## 5. Visual Dashboard & ViT Attention maps
Let's display the final summary dashboard and a couple of attention maps.

In [ ]:
print("Visual summary dashboard:")
display(Image('results/figures/fig9_summary_dashboard.png', width=1000))

print("\nViT Attention Rollout overlay map for OpenCLIP (spices in Africa):")
display(Image('results/figures/attention_openclip_spices_Africa.png', width=800))

print("\nViT Attention Rollout overlay map for SigLIP 2 (spices in Africa):")
display(Image('results/figures/attention_siglip2_spices_Africa.png', width=800))